# Post-Quantum Cryptography & Blockchain

A practical introduction to why quantum computers threaten blockchain cryptography,
and how lattice-based cryptography can secure the future of decentralized systems.


## 1. The Quantum Threat to Public-Key Cryptography

Modern blockchain systems (Bitcoin, Ethereum, etc.) rely on **public-key cryptography**
for digital signatures and key exchange. Bitcoin uses **ECDSA (Elliptic Curve Digital Signature Algorithm)**
based on the secp256k1 curve; Ethereum uses similar ECC schemes.

The security of these systems rests on the **discrete logarithm problem (DLP)**:
given $g$ and $g^x$, it should be computationally infeasible to find $x$. On a classical computer,
the best algorithms for this take exponential time.

### Enter Shor's Algorithm

In 1994, Peter Shor showed that a quantum computer can solve both the **integer factorization problem**
and the **discrete logarithm problem** in **polynomial time** — specifically $O((\log n)^3)$.
This is exponentially faster than any known classical algorithm.

### How Shor's Algorithm Works (Conceptually)

1. **Period finding via QFT**: Shor's algorithm uses the **Quantum Fourier Transform (QFT)**
   to find the period of a function $f(x) = a^x \bmod N$.
2. **Reduction**: Once the period $r$ is found, factoring $N$ or solving DLP reduces to
   simple classical post-processing (gcd calculations).
3. **Impact on ECC**: For elliptic curves, Shor's algorithm can be adapted to find the
   discrete logarithm in $O(n^3)$ time, completely breaking ECDSA and ECDH.

### What This Means for Blockchain

- **Bitcoin addresses** (public-key hash) could have their private key recovered *after* the
  first transaction reveals the public key.
- **All existing signatures** could be forged.
- **Transaction authentication** would be completely compromised.

A quantum computer with ~4000 logical qubits (millions of physical qubits with error correction)
could break 256-bit ECC in hours. While we are not there yet, the threat is serious enough
that NIST and industry leaders are already standardizing replacements.


In [1]:
# Conceptual demonstration: why factoring is hard classically but easy with Shor
import math
import random

def classical_factor(n, max_attempts=100000):
    """Brute-force trial division (classical approach)."""
    for i in range(2, int(math.isqrt(n)) + 1):
        if n % i == 0:
            return i
    return None

# A small "RSA-like" semiprime
p, q = 7919, 7927
N = p * q
print(f"Semiprime N = {p} \u00d7 {q} = {N}")
print(f"Classical trial division finds factor:", end=" ")
factor = classical_factor(N)
print(factor if factor else "not found (too large for brute force)")
print()

# Show the scaling problem
print("Scaling of classical vs quantum factoring:")
for bits in [16, 32, 64, 128, 256]:
    classical_steps = 2 ** (bits // 2)  # approx GNFS sub-exponential
    quantum_steps = bits ** 3           # Shor's polynomial
    print(f"  {bits}-bit key: classical ~2^{bits//2}, quantum ~{bits}³ = {quantum_steps}")


Semiprime N = 7919 × 7927 = 62773913
Classical trial division finds factor: 7919

Scaling of classical vs quantum factoring:
  16-bit key: classical ~2^8, quantum ~16³ = 4096
  32-bit key: classical ~2^16, quantum ~32³ = 32768
  64-bit key: classical ~2^32, quantum ~64³ = 262144
  128-bit key: classical ~2^64, quantum ~128³ = 2097152
  256-bit key: classical ~2^128, quantum ~256³ = 16777216


## 2. Path to Quantum-Resistant Cryptography

To replace ECC and RSA, we need cryptographic primitives based on problems that are **hard for both
classical and quantum computers**.

### What Makes a Problem "Quantum-Safe"?

Shor's algorithm exploits **hidden abelian subgroup structure** in the problem. Problems without this
structure — like certain **NP-complete** problems — are believed to resist quantum attacks.
Grover's algorithm provides a quadratic speedup for unstructured search, but that's manageable
(doubling key sizes restores security).

### The Leading Candidate: Lattice Problems

**Lattices** are regular grids of points in $n$-dimensional space. Two hard problems on lattices:

1. **Shortest Vector Problem (SVP)**: Given a lattice basis, find the shortest non-zero vector.
2. **Learning With Errors (LWE)**: Recover a secret $\mathbf{s}$ from noisy linear equations
   $\mathbf{b} = A\mathbf{s} + \mathbf{e} \pmod{q}$.

These problems are **conjectured hard** for both classical and quantum computers, and reductions
to worst-case lattice problems provide strong theoretical security guarantees.

### Other Quantum-Resistant Families
- **Code-based cryptography** (McEliece): based on decoding random linear codes
- **Multivariate cryptography**: based on solving systems of multivariate polynomial equations
- **Hash-based signatures** (SPHINCS+): based only on the security of hash functions
- **Isogeny-based cryptography**: based on finding isogenies between elliptic curves

NIST's PQC standardization project has selected lattice-based schemes as the primary
recommendation (Kyber for KEM, Dilithium for signatures) due to their balance of
security, performance, and key sizes.


## 3. Lattice-Based Cryptography: Implementing LWE from Scratch

We will now implement a **Learning With Errors (LWE)** encryption scheme using **only basic Python**
(no NumPy, no special libraries). Every function is built from lists, integers, and simple loops.

### The LWE Problem

Given a matrix $A \in \mathbb{Z}_q^{n \times n}$ and a vector $\mathbf{b} = A\mathbf{s} + \mathbf{e}$
where $\mathbf{e}$ is "small" noise, it is **hard** to recover $\mathbf{s}$.
This hardness holds even against quantum computers.

### Our Encryption Scheme

| Step | Operation |
|------|-----------|
| **KeyGen** | Choose $A$, secret $\mathbf{s}$, error $\mathbf{e}$ → $\mathbf{b} = A\mathbf{s} + \mathbf{e}$ |
| **Encrypt** | For bit $m$, choose random $\mathbf{r}$, compute $\mathbf{u} = A^T\mathbf{r}$, $v = \mathbf{b} \cdot \mathbf{r} + m \lfloor q/2 \rfloor$ |
| **Decrypt** | Compute $d = v - \mathbf{s} \cdot \mathbf{u} = \mathbf{e} \cdot \mathbf{r} + m \lfloor q/2 \rfloor$. If $d$ is near $\lfloor q/2 \rfloor$, the bit is 1; otherwise 0. |

The small error $\mathbf{e} \cdot \mathbf{r}$ is negligible compared to $\lfloor q/2 \rfloor$, so decryption works correctly.


In [ ]:
import random

# ═══════════════════════════════════════════
# Lattice helpers — basic math, only lists
# ═══════════════════════════════════════════

def random_vector(n, q=None):
    """Random vector of length n. Entries uniform in [0, q) if q given, else in {0,1}."""
    if q is not None:
        return [random.randrange(0, q) for _ in range(n)]
    return [random.randint(0, 1) for _ in range(n)]

def small_vector(n):
    """Small error vector: entries in {-1, 0, 1}."""
    return [random.choice([-1, 0, 1]) for _ in range(n)]

def dot(a, b):
    """Dot product of two vectors (plain integer, no mod)."""
    return sum(x * y for x, y in zip(a, b))

def mat_vec_mul(A, v, q):
    """Multiply matrix A (m×n) by vector v (n), reduce mod q."""
    return [sum(A[i][j] * v[j] for j in range(len(v))) % q
            for i in range(len(A))]

def transpose(A):
    """Transpose an m×n matrix."""
    return [[A[i][j] for i in range(len(A))] for j in range(len(A[0]))]

def random_matrix(m, n, q):
    """Random m×n matrix with entries uniform in [0, q)."""
    return [[random.randrange(0, q) for _ in range(n)] for _ in range(m)]


In [ ]:
# ═══════════════════════════════════════════
# Key Generation
# ═══════════════════════════════════════════
# Parameters: n = dimension, q = modulus (must be >> n)
n, q = 8, 97

def keygen():
    """
    Generate an LWE key pair.
    
    Public key:  (A, b)  where b = A*s + e (mod q)
    Secret key:   s
    """
    A = random_matrix(n, n, q)          # random public matrix
    s = random_vector(n, q)             # secret key (uniform)
    e = small_vector(n)                 # small error vector
    b = mat_vec_mul(A, s, q)            # b = A*s (mod q)
    b = [(b[i] + e[i]) % q for i in range(n)]  # b = A*s + e
    return (A, b), s

(pk, sk) = keygen()
A, b = pk
print(f"Parameters: n = {n}, q = {q}")
print(f"Public matrix A: {n}×{n} matrix mod {q}")
print(f"Public vector b: {b}")
print(f"Secret key s:    {sk}")


In [ ]:
# ═══════════════════════════════════════════
# Encryption
# ═══════════════════════════════════════════
# Encrypt a single bit m (0 or 1).
# Ciphertext: (u, v) where:
#   u = A^T * r   (mod q)    — hides the randomness
#   v = b·r + m*q//2  (mod q)  — hides the message

def encrypt(pk, bit):
    """Encrypt a single bit (0 or 1) using the LWE public key."""
    A, b = pk
    r = small_vector(n)         # random ephemeral secret (small)
    u = mat_vec_mul(transpose(A), r, q)   # u = A^T * r
    v = (dot(b, r) + bit * (q // 2)) % q  # v = b·r + m*q//2
    return u, v

# Test encryption of both 0 and 1
ct0 = encrypt(pk, 0)
ct1 = encrypt(pk, 1)
print(f"Encrypted bit 0: u = {ct0[0][:3]}..., v = {ct0[1]}")
print(f"Encrypted bit 1: u = {ct1[0][:3]}..., v = {ct1[1]}")


In [ ]:
# ═══════════════════════════════════════════
# Decryption
# ═══════════════════════════════════════════
# Compute d = v - s·u = e·r + m*q//2 (mod q)
# The error e·r is small (bounded by n).
# If d is close to q//2 -> bit = 1
# If d is close to 0    -> bit = 0

def decrypt(sk, ct):
    """Decrypt a ciphertext (u, v) using the LWE secret key."""
    u, v = ct
    d = (v - dot(sk, u)) % q        # d = e·r + m*q//2 (mod q)
    threshold = q // 4
    if threshold < d < q - threshold:
        return 1                     # near q//2
    return 0                         # near 0

print(f"Decrypt(encrypt(0)) = {decrypt(sk, ct0)}")
print(f"Decrypt(encrypt(1)) = {decrypt(sk, ct1)}")


In [ ]:
# ═══════════════════════════════════════════
# Full test: encrypt and decrypt multiple bits
# ═══════════════════════════════════════════
print(f"Testing LWE encryption with n={n}, q={q}")
print("-" * 45)
for bit in [0, 1, 0, 1, 1, 0, 1]:
    ct = encrypt(pk, bit)
    decrypted = decrypt(sk, ct)
    status = "OK" if decrypted == bit else "FAIL"
    print(f"  Original: {bit}  ->  Decrypted: {decrypted}  [{status}]")


## 4. Lattice-Based Cryptography for Blockchain

### Replacing ECDSA with Lattice Signatures

In current blockchains, every transaction is signed with ECDSA. The signature proves
ownership of the private key corresponding to the address.

With lattice-based signatures (like **CRYSTALS-Dilithium** or **Falcon**):
- The **public key** is a lattice matrix $A$ and vector $\mathbf{t} = A\mathbf{s}_1 + \mathbf{s}_2$
- The **signature** is a proof that the signer knows a short vector $\mathbf{s}$ satisfying the equation,
  without revealing $\mathbf{s}$
- Verification checks that the signature is valid AND the vectors are short

### Lattice Key Exchange for Blockchain

Instead of ECDH (Elliptic Curve Diffie-Hellman), two parties can use **LWE key exchange**
to agree on a shared secret. The lattice analogue of Diffie-Hellman works like this:

1. **Alice** generates keys $(A, \mathbf{b}_A), \mathbf{s}_A$ and publishes her public key
2. **Bob** wants to send her an encrypted transaction
3. Bob encrypts the transaction payload using Alice's public key (as we did above)
4. Only Alice can decrypt with her secret key

This directly replaces the ECDH key agreement used in many blockchain protocols.

### Demonstration: Encrypted Blockchain Transaction

Below we simulate a simplified blockchain transaction encrypted with our LWE scheme.
In practice, a real system would use a proper KEM (Key Encapsulation Mechanism) with
IND-CCA2 security and much larger parameters (n > 512, q > 2^12).


In [ ]:
# ═══════════════════════════════════════════
# Lattice-Encrypted Blockchain Transaction
# ═══════════════════════════════════════════
def message_to_bits(msg):
    """Convert a string to a list of bits."""
    bits = []
    for char in msg.encode('utf-8'):
        for i in range(8):
            bits.append((char >> (7 - i)) & 1)
    return bits

def bits_to_message(bits):
    """Convert a list of bits back to a string."""
    chars = []
    for i in range(0, len(bits), 8):
        byte = 0
        for j in range(8):
            byte = (byte << 1) | bits[i + j]
        chars.append(byte)
    return bytes(chars).decode('utf-8')

# Simulate a blockchain transaction
transaction = '{"from": "Alice", "to": "Bob", "amount": 5.0, "nonce": 42}'
print(f"Original transaction: {transaction}")

# Generate a fresh key pair (Alice's wallet)
(alice_pk, alice_sk) = keygen()

# Encrypt the transaction bit by bit
tx_bits = message_to_bits(transaction)
ciphertext = [encrypt(alice_pk, bit) for bit in tx_bits]
print(f"Encrypted {len(tx_bits)} bits into {len(ciphertext)} ciphertext tuples")

# Decrypt
decrypted_bits = [decrypt(alice_sk, ct) for ct in ciphertext]
decrypted_tx = bits_to_message(decrypted_bits)
print(f"Decrypted transaction: {decrypted_tx}")
print(f"Match: {decrypted_tx == transaction}")


## 5. Current Standards: NIST PQC and Google PQC 2026

### NIST Post-Quantum Cryptography Standardization

NIST launched its PQC standardization process in 2016. After multiple rounds of evaluation,
the selected algorithms are:

| Algorithm | Type | Based On | Key Use |
|-----------|------|----------|--------|
| **CRYSTALS-Kyber** | KEM | Module-LWE | Key encapsulation (replaces RSA-KEM, ECDH) |
| **CRYSTALS-Dilithium** | Signature | Module-LWE | General-purpose digital signatures (replaces ECDSA) |
| **FALCON** | Signature | NTRU lattice | Compact signatures for constrained environments |
| **SPHINCS+** | Signature | Hash-based | Stateless hash-based signatures (conservative security) |

In August 2024, NIST released the final FIPS standards:
- **FIPS 203** (CRYSTALS-Kyber / ML-KEM)
- **FIPS 204** (CRYSTALS-Dilithium / ML-DSA)
- **FIPS 205** (SPHINCS+ / SLH-DSA)

### Google PQC 2026 Initiative

Google has committed to migrating its internal infrastructure to PQC by 2026:

- **Chrome** began experimenting with X25519+Kyber hybrid key agreement in 2023
- **Cloud** services offer PQC-enabled TLS (CECPQ2, CECPQ3)
- **Internal infrastructure** targets full PQC migration by 2026
- Google's **BoringSSL** library has integrated Kyber and Dilithium

### Why This Matters for Blockchain

Blockchains are **slow to upgrade** — consensus mechanisms require coordinated hard forks.
Planning PQC migration now is critical:
- **Ethereum** researchers are exploring "quantum-safe" account abstraction
- **Bitcoin** quantum resistance is actively debated (BIP proposals for PQC signatures)
- **New blockchains** (e.g., those built with Substrate or Cosmos SDK) can integrate
  PQC from day one

### References

1. NIST PQC Project: https://csrc.nist.gov/projects/post-quantum-cryptography
2. FIPS 203 (ML-KEM): https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.203.pdf
3. FIPS 204 (ML-DSA): https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.204.pdf
4. FIPS 205 (SLH-DSA): https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.205.pdf
5. Google PQC @ 2026: https://security.googleblog.com/2023/08/transitioning-to-post-quantum.html
6. Shor, P. (1994). "Algorithms for quantum computation: discrete logarithms and factoring"
7. Regev, O. (2005). "On lattices, learning with errors, random linear codes, and cryptography"


## 6. Conclusion

Post-Quantum Cryptography is not a distant problem — **it is happening now**.
NIST has standardized the first wave of PQC algorithms, Google is migrating its
infrastructure, and blockchain projects must begin planning their transitions.

### Key Takeaways

1. **Quantum computers break ECC and RSA** via Shor's algorithm — this is a mathematical
   certainty, not a speculation.
2. **Lattice-based cryptography** (LWE, Module-LWE) is the leading replacement, with
   strong security proofs and excellent performance.
3. **LWE encryption is surprisingly simple** — built from just matrix-vector multiplication
   and small noise, as demonstrated in this notebook.
4. **Blockchain migration is urgent** — the upgrade requires consensus changes and
   should be planned years in advance.
5. **Standards are ready** — NIST FIPS 203/204/205 provide production-ready algorithms.

The transition to PQC will be one of the most significant cryptographic infrastructure
upgrades in history. Starting now gives you a head start.
